# Create Defra Awards (Department for Environment, Food and Rural Affairs)

Creates Defra-sponsored research projects from Defra's own project register,
**Science Search** (https://randd.defra.gov.uk/). Science Search is a Blazor
Server app with no export or API, so `scripts/local/defra_to_s3.py` drives it
with Playwright: the empty search (status = All) lists every project with its
code, title, start/end month and cost; each ProjectDetails page adds the
description, objective, contractor / funded organisations, keywords and fields
of study. **13,640 projects, 1970s-2026, GBP amounts** (local run 2026-09-30).
Defra does not publish investigator names; the first listed contractor is
shipped as `lead_investigator.affiliation` with no person name.

**Overlap check (brief):** no direct-ingest rows exist for F4320319995; UKRI
Gateway to Research and the 360Giving ingests do not carry Defra's funder id.
Existing F4320319995 rows are acknowledgement shells only (~2,270 at priority 0).

**Prerequisites:**
- Run `scripts/local/defra_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/defra/defra_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1):** the Defra project code (`SE3287`, `AC0114`,
`WQ0128`, ...). Citation-side check (`grants.funder:F4320319995` award ids in
`openalex_awards_raw`): the dominant form is exactly this code (`SE3287`,
`AC0114`, `WM0304`, `LS3656`), so our rows hash to the same award id as the
acknowledgement shells and upgrade them in CreateAwards dedup. The internal
Science Search `ProjectId` is kept only in the landing-page URL.

**Amounts:** "Project Cost" in GBP from the result list; a cost of £0 (122
projects) means not recorded and ships as NULL. **Dates:** start / end month
from the result list (start = 1st of month, end = last day of month).

**Funder details (Path A, F4320* Crossref-registered):** funder_id `4320319995`,
metadata read from `openalex.funders.funders` (ror 00tnppw48, doi 10.13039/501100000277).

**Priority:** `500` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.defra_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/defra/defra_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.defra_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.defra_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.defra_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320319995 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320319995;

## Step 2: Create Defra Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.defra_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320319995  -- Department for Environment, Food and Rural Affairs
),

g AS (
    SELECT
        r.*,
        TRY_TO_DATE(r.start_date, 'yyyy-MM-dd') AS sd,
        TRY_TO_DATE(r.end_date, 'yyyy-MM-dd') AS ed,
        NULLIF(TRIM(r.lead_contractor), '') AS contractor
    FROM openalex.awards.defra_raw r
    WHERE r.project_code IS NOT NULL AND TRIM(r.project_code) != ''
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.project_code))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.project_code) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'GBP' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    CAST(NULL AS STRING) as funder_scheme,
    'defra_science_search' as provenance,
    g.sd as start_date,
    g.ed as end_date,
    YEAR(g.sd) as start_year,
    YEAR(g.ed) as end_year,
    -- No person names are published: contractor only, as the affiliation.
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    CASE WHEN g.contractor IS NOT NULL THEN named_struct(
        'given_name', CAST(NULL AS STRING),
        'family_name', CAST(NULL AS STRING),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', g.contractor,
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.project_code))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Shape check (§2.1.1): shipped ids are Defra project codes (letters + digits,
-- no spaces), unique, and never the internal Science Search ProjectId.
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT id) AS distinct_ids,
    SUM(CASE WHEN funder_award_id RLIKE '^[A-Za-z]' AND funder_award_id NOT RLIKE '\\s' THEN 0 ELSE 1 END) AS bad_shape,
    SUM(CASE WHEN landing_page_url LIKE CONCAT('%ProjectId=', funder_award_id) THEN 1 ELSE 0 END) AS shipped_internal_id
FROM openalex.awards.defra_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'defra_science_search' AND priority = 500;

-- Insert into openalex_awards_raw with priority.
-- Priority 500: direct-from-funder ingest of Defra Science Search.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 500 outranks
-- the acknowledgement shells (priority 0) that share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    500 as priority
FROM openalex.awards.defra_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, amount, currency, start_date, end_date,
       lead_investigator.affiliation.name AS contractor
FROM openalex.awards.defra_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_gbp
FROM openalex.awards.defra_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: contractor top-20 (large Defra agencies such as
-- ADAS, Cefas, APHA/VLA legitimately hold hundreds of projects).
SELECT lead_investigator.affiliation.name AS contractor, COUNT(*) AS n
FROM openalex.awards.defra_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator.affiliation.name) as has_contractor
FROM openalex.awards.defra_awards;

In [ ]:
%sql
-- Overlap with the existing acknowledgement shells: how many F4320319995 shell
-- award ids this ingest upgrades (same id).
SELECT
    c.provenance,
    COUNT(*) as shell_rows,
    COUNT(d.id) as upgraded_by_defra_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.defra_awards d ON d.id = c.id
WHERE c.funder_id = 4320319995 AND c.priority = 0
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'defra_science_search'
GROUP BY provenance, priority;